# Databricks


*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Databricks is a managed lakehouse platform built on Apache Spark, Delta Lake, and Unity Catalog; it is where most Spark-and-Delta interview questions become concrete.

## What is Databricks?

Databricks is a managed data platform that packages Apache Spark with Delta Lake, a governance layer (Unity Catalog), a SQL warehouse, workflow orchestration, and ML tooling. The core promise is the **lakehouse**: ACID tables, schema enforcement, and time travel on cheap object storage, with one copy of data serving BI, data science, and streaming.

```text
cloud storage (S3/ADLS/GCS)
        |
   Delta / Iceberg tables          <- open formats, one copy of data
        |
Spark + Photon  |  SQL warehouses  |  MLflow / serving
        |
   Unity Catalog: catalogs.schemas.tables + lineage + grants
```

Interview framing: Databricks is not a database engine you install - it is a control plane plus managed compute on top of your cloud account, with Delta as the table format and Spark as the execution engine.

The mental model that makes every later question easier is three separable layers. The **control plane** is Databricks-managed software: the workspace UI, notebooks, the job scheduler, Unity Catalog metadata, and the cluster manager. The **compute plane** is where code actually executes: virtual machines in your cloud account (classic compute) or in a Databricks-managed account (serverless). The **storage plane** is object storage in your account, where the Parquet files and the Delta log live.

Walk one nightly job through those layers and the billing follows. At 03:00 the control plane's scheduler fires the job and asks the cloud for a job cluster. The driver and executors start as VMs in your account, read Delta files from your bucket, shuffle data over the network, and write new Parquet files plus a `_delta_log` commit. The control plane recorded the run, the Unity Catalog grants were checked, and the SQL warehouse that serves the morning dashboard is separate compute again. You are billed in **DBUs** for Databricks software time and separately by your cloud provider for the VMs, disks, and storage those DBUs ran on (serverless folds both into the DBU rate).

The most common conceptual mistake is assuming Databricks is a database that holds your data. It is not. Databricks sees metadata and query text in the control plane, but your table data sits in your own cloud account and mostly never leaves it. A second mistake is reading a DBU price and treating it as the whole cost: for classic compute the cloud VM bill arrives separately, and for a neglected idle cluster that VM bill keeps running after the notebook is closed. Always ask which of the three planes a cost or a failure belongs to before proposing a fix.

## How does the Databricks architecture work?

+ **Control plane** - the Databricks-managed layer: workspace UI, notebooks, job scheduler, Unity Catalog metadata, cluster manager. Databricks operates it.
+ **Data plane (compute plane)** - clusters and SQL warehouses that run in your cloud account/network, next to your storage. Data does not have to leave your account.
+ **Serverless compute** - Databricks-managed compute in the Databricks account; no cluster to size, faster startup, per-second billing.
+ **Storage** - object storage in your account (S3, ADLS Gen2, GCS), accessed through Unity Catalog external locations and storage credentials rather than hard-coded keys.
+ **Notebooks and jobs** - interactive work and scheduled production work use the same Spark engine and the same tables.

```text
Workspace (control plane)
   |  REST API / CLI / Terraform / bundles
   v
Compute plane (your cloud account)
   +-- all-purpose cluster (interactive, shared)
   +-- job cluster (ephemeral, per-run)
   +-- SQL warehouse (BI and SQL)
   +-- serverless (Databricks account)
   |
   v
Object storage: s3://acme-lake/bronze|silver|gold
```

A workspace is a control-plane object that attaches to exactly one Unity Catalog **metastore** for a region, and many workspaces in that region share it. That is why a table created in one workspace is immediately visible in another attached workspace, and why `USE CATALOG` resolves to the same three-level namespace everywhere. Serverless compute changes the compute plane rather than the control plane: Databricks runs the VMs in its own account and connects to your storage through a storage credential, optionally over a private endpoint.

A useful test is to point at any component and say which plane owns it. The notebook file and its revision history: control plane. The Spark driver process, the executors, and the shuffle files on local disk: compute plane. The `_delta_log` JSON commits and the Parquet files: storage plane. The Unity Catalog grant that let the job read `acme.silver.orders`: control-plane metadata, enforced at query time. If a bucket is unreachable, the compute plane is failing to reach the storage plane, and no amount of cluster resizing helps.

The classic version of this mistake is putting credentials in a notebook. A developer stores an access key in a widget or an environment variable so `spark.read` can reach the bucket, and it works until the key rotates or an auditor asks who can read the data. The correct path is a cloud identity behind a **storage credential**, wrapped by an **external location**, with Unity Catalog deciding who may use it. The notebook then reads a governed path and never holds a secret.

## What cluster types exist and when do you use each?

| Cluster | Lifetime | Used for | Cost profile |
|---|---|---|---|
| All-purpose | long-lived, shared | interactive notebooks, exploration, ad-hoc | expensive per DBU; idle cost if not auto-stopped |
| Job cluster | created per job run | production pipelines | cheapest for scheduled work, isolated per run |
| SQL warehouse | managed, auto-stop | BI, dashboards, SQL users | sized by warehouse size and clusters |
| Serverless (jobs/SQL/notebooks) | managed | bursty or latency-sensitive workloads | no cluster management, per-second billing |

Rules of thumb:

+ Run production pipelines on **job clusters** (or serverless jobs), never on a developer's all-purpose cluster.
+ Use **all-purpose** clusters for exploration and pair them with aggressive auto-stop.
+ Use **SQL warehouses** for BI so warehouse sizing and concurrency are independent of Spark jobs.
+ Give each workload its own cluster **access mode** and policy when governance requires isolation.

```json
{
  "cluster_name": "etl-daily",
  "spark_version": "auto:latest-lts",
  "node_type_id": "m5d.4xlarge",
  "autoscale": {"min_workers": 4, "max_workers": 32},
  "aws_attributes": {"availability": "SPOT_WITH_FALLBACK", "zone_id": "auto"},
  "autotermination_minutes": 20,
  "custom_tags": {"team": "data-platform", "cost-center": "analytics"},
  "spark_conf": {"spark.sql.adaptive.enabled": "true"}
}
```

The four compute types differ less in the engine than in lifetime and billing. An **all-purpose cluster** is created once and reused by many people, so it bills the premium all-purpose DBU rate for every minute it is up, including idle minutes. A **job cluster** is created for one run and destroyed at the end, so it bills the cheaper jobs rate and never idles. A **SQL warehouse** is tuned for SQL concurrency and result serving, with its own sizing and auto-stop. **Serverless** compute removes cluster management entirely and bills per second, with startup measured in seconds rather than minutes.

Choose by the shape of the workload, not by habit. Interactive exploration, debugging, and notebooks people sit in front of: all-purpose with aggressive auto-stop. Scheduled pipelines and backfills: job clusters or serverless jobs, one per run, isolated from each other. BI dashboards and SQL editors: a SQL warehouse so concurrency is managed independently of Spark. Bursty or spiky work where cluster startup dominates: serverless. A useful default for a new pipeline is a job cluster with Photon and a range of workers, and a matching serverless variant once the runtime is stable.

The mistake that shows up in cost reviews is production work living on a shared all-purpose cluster. Two engineers exploring at the same time and one scheduled job are now coupled: a runaway notebook slows the pipeline, and one forgotten session keeps the premium DBU rate running all weekend. The mirror-image mistake is using a SQL warehouse for Spark work or a general cluster for a dashboard; each is sized and priced for its own access pattern, and moving the workload to the right type is often the single largest saving available.

## How do you size and configure a cluster for performance and cost?

+ **Workers** - memory-optimized (r-family) for shuffles and caching; compute-optimized (c-family) for CPU-heavy transformation; storage-optimized (i/d-family) when spilling or caching to local disk matters.
+ **Autoscaling** - scale within a range; it reacts to pending tasks, so it is not a substitute for fixing skew.
+ **Spot instances with fallback** - large savings for batch; avoid for latency-critical or stateful streaming drivers.
+ **Driver size** - the driver collects results and runs the DAG; `collect()` on a large DataFrame still kills the driver.
+ **Photon** - enable it for SQL/DataFrame heavy workloads; it does not accelerate row-at-a-time Python UDFs.
+ **Auto-stop** - set 10-20 minutes on interactive clusters; the biggest source of accidental spend is an idle cluster.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Check what a job actually used: sizing questions are answered with data, not opinions
display(spark.sql("""
    SELECT usage_metadata.cluster_id AS cluster_id,
           usage_metadata.job_id AS job_id,
           SUM(usage_quantity) AS dbus
    FROM system.billing.usage
    WHERE usage_date >= current_date() - INTERVAL 30 DAYS
    GROUP BY usage_metadata.cluster_id, usage_metadata.job_id
    ORDER BY dbus DESC
    LIMIT 20
"""))

Node families trade CPU, memory, and local disk against each other. **Memory-optimized** instances (the r-family on AWS) carry more RAM per core, which suits large shuffles, joins, aggregations, and cached DataFrames. **Compute-optimized** instances (c-family) carry more cores per unit of RAM and suit CPU-bound parsing, encoding, and narrow transformations. **Storage-optimized** instances (i- and d-families) add fast local NVMe disk, which matters when shuffle spill or disk caching is the bottleneck. A cluster is one driver plus N workers, and each worker runs exactly one executor, so worker count and worker size together set total parallelism.

Autoscaling adjusts the worker count inside a min/max range based on pending tasks, and the modern behaviour is smarter than a naive queue check: it can scale up from min to max in at most two events, and it can scale down while work is still running by looking at the shuffle file state. On job compute it considers the cluster underutilized after roughly 40 seconds and on all-purpose after roughly 150 seconds. Do not also enable Spark dynamic allocation, because the two managers fight and produce executor churn. Spot instances cut VM cost, but the driver is always on-demand and failback to on-demand only applies to spot instances that were fully acquired and running.

Driver sizing is the setting people forget. The driver runs the DAG, collects results, and holds the state of every attached notebook, so a `collect()` or `toPandas()` on a large result kills the driver while the workers sit idle. Size the driver for coordination and result handling, and use a larger driver when a notebook legitimately pulls many rows back. The corresponding mistake is expecting autoscaling to fix skew: a range of workers cannot split one hot partition, so the job still has one straggler task no matter how high the maximum is.

## What is Unity Catalog and how does the three-level namespace work?

Unity Catalog is the governance layer for data and AI assets. Objects are addressed with three levels:

```text
catalog.schema.table
acme.silver.orders
acme.gold.fct_orders
acme.bronze.orders_raw
```

+ **Metastore** - one per region (typically); attaches to workspaces.
+ **Catalog** - the top-level container, often per environment (`dev`, `prod`) or per domain.
+ **Schema** - database-level grouping of tables, views, volumes, functions, models.
+ **Volumes** - governed storage for files that are not tables (`/Volumes/acme/landing/orders/`).
+ **External locations and storage credentials** - central place to grant access to cloud storage; notebooks no longer need access keys.

```sql
CREATE CATALOG IF NOT EXISTS acme;
CREATE SCHEMA IF NOT EXISTS acme.silver;

CREATE TABLE IF NOT EXISTS acme.silver.orders (
    order_id      STRING NOT NULL,
    customer_id   STRING,
    order_ts      TIMESTAMP,
    amount        DECIMAL(18, 2),
    status        STRING
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
```

Unity Catalog replaces the two-level `database.table` namespace with `catalog.schema.table`. A **metastore** is the top container and is attached to workspaces in one region. A **catalog** is the unit that usually maps to an environment or a domain (`dev`, `prod`, `finance`), a **schema** groups tables, views, volumes, functions, and models, and the object name is resolved left to right. Because the catalog is part of the name, the same physical cluster can read `dev.silver.orders` and `prod.silver.orders` and the grants on each are completely independent.

File access follows a deliberate chain. A cloud identity (an IAM role or a managed identity) is registered as a **storage credential**. An **external location** names a storage path and points at that credential. Only then can a table or volume be created at that path. The grants mirror the chain: `CREATE EXTERNAL LOCATION` on the metastore, `CREATE EXTERNAL TABLE` or `CREATE EXTERNAL VOLUME` on the location, and `READ FILES` or `WRITE FILES` on the location for direct path access. Managed tables, by contrast, are created under a managed location and need no per-path grant.

The two habits that break here are hard-coded cloud paths with embedded keys, and two-level names carried over from Hive. A job that reads `s3://acme-raw/orders/` with a key in the notebook bypasses every governance control; the same job reading an external location path inherits them. And after migration, `SELECT * FROM silver.orders` no longer resolves because the default catalog or schema is wrong, so production jobs should always use fully qualified `catalog.schema.table` names rather than relying on session defaults.

## How do you govern data with Unity Catalog?

Governance is a first-class interview topic: grants, fine-grained filters, and lineage all live in the catalog.

```sql
-- role-style grants at catalog/schema/table level
GRANT USE CATALOG ON CATALOG acme TO `analytics`;
GRANT USE SCHEMA ON SCHEMA acme.gold TO `analytics`;
GRANT SELECT ON TABLE acme.gold.fct_orders TO `analytics`;

-- column mask: analysts see a hashed email
CREATE FUNCTION acme.security.mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('pii-readers') THEN email
            ELSE sha2(email, 256) END;

ALTER TABLE acme.gold.dim_customer
ALTER COLUMN email SET MASK acme.security.mask_email;

-- row filter: each regional team sees only its region
CREATE FUNCTION acme.security.region_filter(region STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('global-readers')
       OR is_account_group_member(CONCAT('region-', region));

ALTER TABLE acme.gold.fct_orders
SET ROW FILTER acme.security.region_filter ON (region);
```

Also know **attribute-based access** with tags, **lineage** (table -> notebook/job -> dashboard), **audit logs** in `system.access.audit`, and **data contracts** via constraints plus enforced schemas.

A **row filter** is a SQL UDF that returns a boolean and is evaluated per row at query time; `ALTER TABLE ... SET ROW FILTER func ON (col)` binds it to a table, and rows where it returns false are invisible. A **column mask** is a SQL UDF that receives a column value and returns a value of the same type, bound with `ALTER TABLE ... ALTER COLUMN col SET MASK func`. Both are checked in the engine, so a user with `SELECT` still sees only the filtered rows and the masked values. The common pattern uses `is_account_group_member('group')` inside the function to branch on the caller.

The performance cost is real because the engine must not let a user infer hidden values from the query plan. That means predicate pushdown and some optimizations are restricted when a filter or mask is present. Keep the UDFs simple, prefer a `CASE` expression over a lookup table or a subquery, apply masks only to columns that truly need them, and reduce the number of UDF arguments because the engine cannot drop column references that come from arguments. Watch the type signatures too: with ANSI mode off, a `STRING` column passed to an `INT` parameter is silently cast to null, and a filter written as `dept IS NULL` then returns every row instead of none.

The grant model is hierarchical and additive. `GRANT USE CATALOG` lets a principal address the catalog, `GRANT USE SCHEMA` lets it address the schema, and `GRANT SELECT` or `GRANT MODIFY` grants the actual operation; missing an intermediate `USE` privilege is why a `GRANT SELECT` on a table can appear to do nothing. `MANAGE` delegates the ability to grant on an object, and `ALL PRIVILEGES` covers the object it is run on plus the `USE` privileges below it. For centralized rules that cover many tables at once, attribute-based policies attached to a catalog or schema are now the recommended alternative to per-table filters and masks.

## How does Delta Lake work on Databricks?

Delta Lake is the default table format on Databricks: Parquet files plus a transaction log (`_delta_log`) that adds ACID commits, schema enforcement, time travel, and efficient updates/deletes.

```sql
-- time travel and restore
SELECT * FROM acme.silver.orders VERSION AS OF 42;
SELECT * FROM acme.silver.orders TIMESTAMP AS OF '2026-10-01T00:00:00Z';
RESTORE TABLE acme.silver.orders TO VERSION AS OF 42;

-- inspect history and table details
DESCRIBE HISTORY acme.silver.orders;
DESCRIBE DETAIL acme.silver.orders;
```

In [ ]:
# ILLUSTRATIVE - requires delta: install to run, or read on.
from delta.tables import DeltaTable

# idempotent upsert from a staging DataFrame
(
    DeltaTable.forName(spark, "acme.silver.orders").alias("t")
    .merge(staged.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
    .whenNotMatchedInsertAll()
    .execute()
)

Every write to a Delta table appends a JSON commit to `_delta_log` that lists the files added and removed. Readers resolve the current snapshot by replaying the log, and roughly every ten commits Delta writes a Parquet checkpoint so replay stays cheap. Because a commit is a rename into the log, a write either fully lands or does not, which is what gives Delta its ACID guarantees on object storage. Concurrent writers use optimistic concurrency: they commit against the version they read and retry if someone else committed first, and features such as deletion vectors and row tracking let disjoint changes proceed with row-level concurrency.

Time travel reads the same log at an older version. `VERSION AS OF 42` or `TIMESTAMP AS OF '...'` gives the snapshot as of that point, and `RESTORE TABLE ... TO VERSION AS OF` makes an old version current again. The window is bounded by two retentions: log files are kept about 30 days, and data files are removed by `VACUUM` after `delta.deletedFileRetentionDuration`, which defaults to 7 days. Running `VACUUM` inside the window you rely on is irreversible, so resolve the retention policy before anyone reaches for it during an incident.

`MERGE` is the upsert primitive: match source rows to target rows on a condition, then update, insert, or delete. It is idempotent when the source is deterministic and has one row per key, which is what makes reruns safe. A source with duplicate keys raises an error by default because the outcome would be ambiguous, so deduplicate upstream or in a staging view. On tables with deletion vectors the old files are not rewritten and `REORG TABLE ... APPLY (PURGE)` is what physically applies the soft deletes before a later `VACUUM` removes the expired files.

## How do you run incremental ingestion with Auto Loader?

Auto Loader (`cloudFiles`) incrementally discovers new files in cloud storage, tracks them in a RocksDB-backed state store, and handles schema inference and evolution. It is the standard bronze-layer ingestion pattern.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
schema_location = "/Volumes/acme/bronze/checkpoints/orders_schema"
checkpoint = "/Volumes/acme/bronze/checkpoints/orders"

raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.maxFilesPerTrigger", 1000)
    .load("s3://acme-raw/orders/")
)

(
    raw.writeStream
    .option("checkpointLocation", checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)          # batch-style catch-up, then stop
    .toTable("acme.bronze.orders_raw")
)

Why it matters: no file-listing bookkeeping, exactly-once progress via checkpoints, and new columns do not break the pipeline. `availableNow` gives you scheduled micro-batch behaviour without keeping a streaming cluster alive.

Auto Loader discovers new files in cloud storage and keeps track of which ones it has already processed. It can use cloud file-notification services so the source pushes events, or fall back to a directory listing, and either way progress lives in a RocksDB-backed state store keyed by a stream checkpoint. A second, separate directory holds the inferred schema over time. New columns land in the schema location as they appear, and because the checkpoint and the schema location are different things, deleting one while keeping the other produces confusing failures rather than a clean reset.

Schema evolution is driven by `cloudFiles.schemaEvolutionMode`. The default, `addNewColumns`, adds the new column to the stored schema and then stops the stream with an error on the micro-batch, expecting a restart to continue with the widened schema; `addNewColumnsWithTypeWidening` also widens compatible types such as int to long; `rescue` never evolves the schema and puts unexpected fields in the `_rescued_data` column; `failOnNewColumns` stops and waits for a human; and `none` ignores new columns entirely. Because the default fails fast by design, production streams should run under a job with retries so the restart happens automatically. `availableNow=True` runs the stream as a catch-up batch and stops, which is how a scheduled pipeline gets streaming semantics without keeping a cluster alive.

The mistakes cluster around state. Two ingestion streams pointing at the same checkpoint interfere, so each source location needs its own checkpoint and schema location. Setting `cloudFiles.maxFilesPerTrigger` or `cloudFiles.maxBytesPerTrigger` too high produces a first micro-batch that looks like a full backfill and can take hours. And a rescued-data column is only a safety net: rows still deserve an alert, because otherwise type mismatches accumulate silently in `_rescued_data` while the typed column reads null.

## How do you build a medallion pipeline (bronze, silver, gold)?

The medallion pattern separates raw landing from conformed data from serving tables:

```text
bronze  raw append-only, Auto Loader, no business logic, keeps history
  |
silver  cleaned, deduplicated, typed, conformed keys; MERGE upserts
  |
gold    facts, dimensions, aggregates that BI and ML read
```

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# silver: dedupe raw events, then merge into a conformed table
from pyspark.sql import Window, functions as F

raw = spark.readStream.table("acme.bronze.orders_raw")

deduped = (
    raw.withWatermark("ingested_at", "2 hours")
    .dropDuplicates(["order_id"])
)

(
    deduped.writeStream
    .option("checkpointLocation", "/Volumes/acme/checkpoints/silver_orders")
    .trigger(availableNow=True)
    .foreachBatch(merge_into_silver)
    .toTable("acme.silver.orders")
)

Keep bronze immutable (replayability), put quality expectations in silver, and keep gold small enough for fast dashboard queries.

The medallion pattern is a contract about what each layer is allowed to contain. Bronze is raw and append-only: the bytes that arrived, plus ingestion metadata, with no business rules. Silver is conformed: deduplicated, typed, validated, joined to reference data, and kept at the grain the business describes. Gold is shaped for consumption: facts, dimensions, and aggregates that BI and ML read directly. Each layer is a Delta table, so each layer has ACID commits, a history, and a schema that can be inspected.

The layering buys replayability. If a silver transformation has a bug, you fix the code and rebuild silver from bronze, because bronze still holds everything. If you instead deleted the bad rows in bronze, the evidence is gone and the fix becomes a manual data repair. A concrete run looks like this: Auto Loader appends raw JSON to `acme.bronze.orders_raw`, a silver merge deduplicates on `order_id` into `acme.silver.orders`, and a gold job groups that into `acme.gold.fct_daily_revenue`. Only the gold table is small enough to serve a dashboard in under a second.

Two mistakes recur. Putting business logic in bronze destroys the raw record and makes reprocessing meaningless. Making gold a full copy of silver, with every column and every row, means the dashboard pays silver's scan cost with none of silver's flexibility; gold should be narrower and pre-aggregated to what is queried. A third, quieter one is letting bronze grow forever with no retention or archival plan, which eventually costs more in storage and file listing than the pipeline saves.

## What are Lakeflow Declarative Pipelines (Delta Live Tables)?

Lakeflow Declarative Pipelines (formerly Delta Live Tables) let you declare datasets and expectations; the runtime handles dependency ordering, incremental processing, retries, and data-quality metrics.

In [ ]:
# ILLUSTRATIVE - requires dlt, pyspark: install to run, or read on.
import dlt                      # older runtimes expose dlt; newer ones also expose pyspark.pipelines
from pyspark.sql import functions as F

@dlt.table(name="bronze_orders", comment="Raw orders from cloud storage")
def bronze_orders():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/acme/dlt/orders_schema")
        .load("s3://acme-raw/orders/")
    )

@dlt.table(name="silver_orders")
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_fail("valid_amount", "amount >= 0")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("order_date", F.to_date("order_ts"))
        .dropDuplicates(["order_id"])
    )

Interview points: expectations are declared with the table (not in a separate tool), `expect_or_drop` quarantines bad rows while `expect_or_fail` stops the pipeline, and both streaming tables and materialized views are supported.

A Lakeflow Declarative Pipeline is a set of dataset definitions, each a function decorated to say what it produces, and the runtime works out the dependency graph from the reads and writes in those functions. You do not schedule individual steps; you declare tables and the engine orders, parallelizes, retries, and records the run. The two dataset kinds are the **streaming table**, which processes each batch incrementally and appends, and the **materialized view**, which recomputes a query result and is right when the result is an aggregate over a changing table. Pipelines run on serverless or classic compute, and the newer runtimes expose the API as `from pyspark import pipelines as dp` alongside the older `import dlt`.

Expectations attach quality rules to a definition and are evaluated as data flows. `expect` records a metric and keeps every row, `expect_or_drop` removes failing rows and counts them, and `expect_or_fail` stops the update. The metrics show up per table in the pipeline event log, so data quality is visible without a separate tool. The other thing to get right is batch versus streaming reads inside a definition: `read_stream` preserves incremental semantics, while `read` materializes a full snapshot, and mixing them carelessly turns an incremental pipeline into a full recompute on every run.

The mistakes are mostly about expectations and refresh. Dropping bad rows without routing them anywhere means the failures disappear from the table and live only in a metric, so pair `expect_or_drop` with a quarantine table when the data matters. Choosing a materialized view where a streaming table belongs forces a full source scan each update. And expecting expectations to be shared across pipelines is wrong: they belong to the definition that declares them, which is a feature for locality and a problem if you wanted one org-wide rule set.

## How do you orchestrate jobs and workflows?

Databricks Workflows run multi-task DAGs with dependencies, schedules, triggers, retries, and alerts.

In [ ]:
# ILLUSTRATIVE - requires requests: install to run, or read on.
# Jobs API: submit a task-based run (simplified)
import requests

payload = {
    "run_name": "daily-orders",
    "tasks": [
        {"task_key": "bronze", "notebook_task": {"notebook_path": "/Repos/prod/notebooks/bronze"}},
        {"task_key": "silver", "depends_on": [{"task_key": "bronze"}],
         "notebook_task": {"notebook_path": "/Repos/prod/notebooks/silver"}},
        {"task_key": "gold", "depends_on": [{"task_key": "silver"}],
         "sql_task": {"warehouse_id": "abc123", "query": {"query_id": "gold-refresh"}}},
    ],
    "schedule": {"quartz_cron_expression": "0 0 3 * * ?", "timezone_id": "UTC"},
    "max_concurrent_runs": 1,
}
requests.post("https://acme.cloud.databricks.com/api/2.1/jobs/create",
              headers={"Authorization": f"Bearer {token}"}, json=payload, timeout=30)

Useful features to name: **file-arrival triggers**, **task values** for passing small results between tasks, **retry policies per task**, **job parameters**, **repair runs** (re-run only failed tasks), and **notifications** on failure or duration.

A Workflow is a job made of tasks with dependencies, expressed as a directed acyclic graph. Each task has a key, a type (notebook, Python wheel, SQL, pipeline, and others), optional `depends_on` entries, and its own cluster or an inherited one. Task values let one task publish a small result that a downstream task reads as `{{tasks.<key>.values.<name>}}`, and job parameters flow into every task that asks for them. Because dependencies are explicit, unrelated tasks run in parallel and the run is reproducible from the job definition alone.

Triggers decide when the graph runs: a cron schedule with a time zone, a file-arrival trigger on a storage path, a table-update trigger, or continuous mode for always-on streaming. Each task can have a retry policy with a maximum number of attempts and an interval, plus a timeout, and failures can raise notifications or a webhook. **Repair runs** re-execute only the tasks that failed or were skipped, reusing the successful tasks' outputs, which is what makes a long DAG cheap to recover. `max_concurrent_runs` caps overlap so a slow run and the next schedule do not fight over the same tables.

The mistakes are easy to name and common in practice. Leaving `max_concurrent_runs` unset lets a 03:00 run still be going when the 04:00 run starts. Passing a large DataFrame through task values fails, because values are meant for small scalars and flags, not data. Re-running the whole job instead of using a repair run wastes hours and can double-write tables. And putting the cluster definition in every task rather than using a shared job cluster makes a DAG slow to start and harder to reason about.

## How do you manage Databricks resources as code?

Databricks Asset Bundles (DABs) package jobs, pipelines, notebooks, and config into a repo deployable with the CLI; Terraform manages workspace-level infrastructure.

```yaml
# databricks.yml
bundle: acme_lakehouse
workspace:
  host: https://acme.cloud.databricks.com
variables:
  catalog:
    default: acme_dev
resources:
  jobs:
    daily_orders:
      name: daily-orders-${bundle.target}
      schedule:
        quartz_cron_expression: "0 0 3 * * ?"
        timezone_id: UTC
      tasks:
        - task_key: bronze
          notebook_task:
            notebook_path: ./notebooks/bronze_orders.py
        - task_key: silver
          depends_on:
            - task_key: bronze
          notebook_task:
            notebook_path: ./notebooks/silver_orders.py
```

```bash
databricks bundle validate -t prod
databricks bundle deploy -t prod
databricks bundle run daily_orders -t prod
```

```hcl
# Terraform for workspace-level resources
resource "databricks_job" "daily_orders" {
  name = "daily-orders"
  task {
    task_key = "bronze"
    notebook_task { notebook_path = "/Repos/prod/notebooks/bronze" }
  }
}
```

Databricks Asset Bundles are the first-party way to keep jobs, pipelines, and their supporting files in a Git repository. A `databricks.yml` declares the bundle, one or more targets such as `dev` and `prod`, variables, and the resources. `databricks bundle validate` checks the configuration and resolves references without deploying; `databricks bundle deploy -t prod` syncs the files and creates or updates the resources; `databricks bundle run <resource> -t prod` triggers them. Development targets can prefix names and pause schedules so a test deploy does not start writing to production tables.

Bundles cover workspace-level resources, and Terraform covers the account-level ground beneath them: metastores, storage credentials, external locations, catalogs, cluster policies, service principals, and workspace assignment. The two are complementary rather than competing, and the rule of thumb is to let Terraform own long-lived governance objects and let bundles own the jobs and pipelines that change every week. Terraform state is the thing to protect: store it remotely with locking, and never let two people apply the same state from a laptop.

The classic bundle mistake is making changes in the workspace UI and then wondering why the next `bundle deploy` reverts them; the repository is the source of truth, so either move the change into the bundle or let it be overwritten. The second is committing secrets or workspace hosts into `databricks.yml`; hosts are fine as variables, secrets belong in a secret scope or in CI variables. The third is letting a bundle and a Terraform resource both manage the same job, which produces a fight where each tool believes it is authoritative.

## How do you query data with Databricks SQL?

SQL warehouses are separate compute for SQL workloads, with serverless or classic sizing, auto-stop, and query federation to external systems.

```sql
-- cost and performance introspection from system tables
SELECT workspace_id, compute.warehouse_id AS warehouse_id,
       SUM(total_duration_ms) / 1000 AS seconds
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 7 DAYS
GROUP BY workspace_id, compute.warehouse_id
ORDER BY seconds DESC;

-- liquid clustering + maintenance on a gold table
ALTER TABLE acme.gold.fct_orders CLUSTER BY (order_date, customer_id);
OPTIMIZE acme.gold.fct_orders;
```

Also relevant: **Databricks SQL dashboards and alerts**, **query federation** to other databases, **materialized views** for repeated aggregations, and **Photon** as the default for SQL warehouses.

A SQL warehouse is serverless or classic SQL compute with a size chosen from a T-shirt scale, from 2X-Small up to 4X-Large, and a cluster count that can scale horizontally. Increasing the size adds nodes so one query runs faster; increasing the maximum cluster count lets more queries run at once. A result cache serves an identical query over unchanged tables without re-executing it, and an IO cache keeps recently read data local for repeat scans. Serverless warehouses remove the sizing decision for the compute layer and start quickly, while classic warehouses give you the sizing controls and network placement you may require.

Two scaling decisions sit on top of the size. Multi-cluster scaling adds identical clusters behind one endpoint when queries queue, so concurrency does not force one enormous warehouse; auto-stop suspends the endpoint after an idle interval so a dashboard nobody opened overnight costs nothing. Materialized views precompute a repeated aggregate and refresh incrementally, which is often better than making every dashboard hit the base table. Query federation lets a warehouse read external databases through a connection, which is useful for migrations and joins against systems that have not moved yet.

The expensive mistake is one giant warehouse for everything: dashboard point queries, a nightly extract, and ad hoc analysis share a size chosen for the heaviest one, and the auto-stop is set long because someone is always using it. Separate warehouses per workload, each sized for its own queries and auto-stopped aggressively, is cheaper and isolates a runaway query from the dashboard. The other mistake is assuming the result cache always applies; any write to a source table, or a changed session parameter that is part of the cache key, invalidates it, and different query text is always a miss.

## What is Photon and when does it help?

Photon is a vectorized C++ query engine that replaces the JVM execution path for supported operations in Spark and SQL workloads.

+ **Helps**: large scans, filters, joins, aggregations, Delta MERGE, SQL warehouses.
+ **Does not help**: Python UDFs and row-at-a-time code, which fall back to the slower path.
+ **Trade-off**: Photon clusters cost more per DBU but are often cheaper per job because they finish faster - validate with the query profile rather than assuming.

```text
Rule of interview thumb: replace Python UDFs with built-in functions, then measure;
Photon cannot rescue row-by-row Python.
```

Photon is a vectorized C++ execution engine that replaces the JVM path for the operators it supports. Catalyst still plans the query, but Photon processes data in columnar batches of thousands of rows, which lets the CPU use SIMD instructions, removes garbage-collection pauses, and yields cache-friendly sequential reads. Supported operators include scans over Parquet, Delta, CSV, and JSON, filters, projections, hash joins and aggregations, shuffles, sorts, windows, unions, and the Delta and Parquet write sink, plus a wide set of expressions and types. When a query hits something unsupported, that operation falls back to Spark transparently and the results stay correct.

Photon does not help every workload, and knowing the exceptions is the point of the question. It does not support **UDFs**, **RDD APIs**, or **Dataset APIs**, so a job whose hot path is a row-at-a-time Python UDF pays the Photon DBU premium while that stage runs on the slow path. It does not support **stateful streaming**; only stateless streaming to a Delta or Parquet sink is accelerated. It also does not meaningfully help queries that finish in under about two seconds, because those are dominated by planning and scheduling rather than data processing. Heavy client-side work on the driver, such as collecting a large result or looping in Python, is untouched by Photon no matter how many workers are attached.

Verify before assuming. On classic compute, the Spark UI SQL/DataFrame tab draws Photon operators in orange and standard Spark operators in blue, so a mostly blue plan explains a disappointing speedup. On SQL warehouses and serverless, the query profile shows the percentage of task time spent in Photon and colours Photon operators differently. In billing, `product_features.is_photon` in `system.billing.usage` tells you whether a usage record was Photon, which is how you prove a Photon experiment paid off rather than guessing from wall-clock time.

## How do you tune Delta tables on Databricks?

+ **File size** - target roughly 100 MB-1 GB per file; small files kill scans and metadata.
+ **Liquid clustering** - the current recommended layout strategy: `CLUSTER BY (order_date, customer_id)` lets Databricks manage file layout incrementally and supports changing the keys.
+ **Partitioning** - still valid when a low-cardinality column is always filtered (`date`), but avoid over-partitioning; do not combine manual partitioning with clustering on the same table.
+ **ZORDER** - legacy co-location technique; still works, but liquid clustering is preferred for new tables.
+ **OPTIMIZE / VACUUM** - compact files and reclaim space; VACUUM cannot go inside the time-travel retention window.
+ **Predictive optimization** - Databricks can run OPTIMIZE/VACUUM/ANALYZE automatically for eligible tables.

```sql
ALTER TABLE acme.silver.orders CLUSTER BY (order_date, customer_id);
OPTIMIZE acme.silver.orders;
VACUUM acme.silver.orders RETAIN 168 HOURS;
ANALYZE TABLE acme.silver.orders COMPUTE STATISTICS FOR ALL COLUMNS;
```

File layout is the first lever because it decides how much data a query must open. Delta keeps per-file statistics, including minimum and maximum values for the first 32 columns, and uses them to skip files whose ranges cannot match a predicate; that is **data skipping**. The target is roughly 100 MB to 1 GB per file. Many small files multiply the number of opens, the amount of metadata, and the number of tasks, while a few enormous files limit how much parallelism a scan can use and make rewrites expensive. `OPTIMIZE` compacts and clusters, and `VACUUM` removes files no longer referenced by the current table.

Three layout techniques now compete. **Liquid clustering**, declared with `CLUSTER BY (col1, col2)`, is the recommended default for new tables: keys can be changed later without rewriting history, `OPTIMIZE` reclusters incrementally, and `CLUSTER BY AUTO` lets predictive optimization choose keys from the query workload. **Partitioning** still works when a genuinely low-cardinality column is filtered by almost every query, but over-partitioning creates small files and metadata bloat, and it is rigid. **ZORDER** is the older co-location technique and remains readable, but it is not compatible with clustering. Liquid clustering and partitioning cannot be combined on the same table, so convert rather than stack.

The mistakes are predictable. Partitioning by a high-cardinality column such as `customer_id` produces millions of tiny directories. Partitioning by a column nobody filters buys nothing and costs write overhead. Changing `CLUSTER BY` keys without running `OPTIMIZE ... FULL` leaves existing data laid out for the old keys, so the change looks ineffective. And scheduling an hourly `OPTIMIZE` while predictive optimization is also enabled means two systems doing the same maintenance, which usually pays twice for the same work.

## How do you handle schema evolution and Change Data Feed?

```sql
-- merge schema on ingest for new columns
SET spark.databricks.delta.schema.autoMerge.enabled = true;

-- change data feed lets downstream jobs read only changed rows
ALTER TABLE acme.silver.orders SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

SELECT order_id, _change_type, _commit_version, order_ts, amount
FROM table_changes('acme.silver.orders', 100)
WHERE _change_type IN ('insert', 'update_postimage');
```

Rules that prevent incidents: add columns as nullable, never reuse a column name for a new meaning, and treat renames as migrations (backfill plus view) rather than schema evolution.

Change Data Feed records row-level changes so a downstream job can read only what changed. With `delta.enableChangeDataFeed = true`, reads through `table_changes('catalog.schema.table', start_version)` return each changed row tagged with `_change_type` as `insert`, `update_preimage`, `update_postimage`, or `delete`, alongside `_commit_version` and `_commit_timestamp`. The feed covers a version range or a timestamp range, which makes it the natural source for incremental silver and gold layers instead of rescanning the whole table. Its history is bounded by `VACUUM`, because the change data files live under the table directory and are removed with the expired data files.

Schema evolution is the other half of a stable pipeline. `mergeSchema` lets a write add new columns from the incoming data, and `spark.databricks.delta.schema.autoMerge.enabled` extends that to merges and updates within a statement. Column mapping decouples the physical Parquet column names from the logical names, which is what makes a rename possible without rewriting every file. Type widening lets a compatible type change, such as int to long, be recorded as metadata. All of these are one-way doors for older readers: once a table's protocol is upgraded, a runtime that does not support the feature cannot read it at all.

The rules that prevent incidents are simple. Add columns as nullable, because adding a required column to a populated table cannot be satisfied retroactively. Never reuse a name for a new meaning; a rename is a migration with a backfill and a compatibility view, not schema evolution. Do not turn on Change Data Feed after the fact and expect history, because only changes committed while the property was enabled are recorded. And do not `VACUUM` aggressively on a table whose downstream consumers read changes from three days ago.

## How do you implement CI/CD for Databricks?

```text
1. Repos: notebooks and jobs live in Git (Databricks Repos or external CI checkout).
2. Validate: databricks bundle validate in each PR; run unit tests with pytest on pure PySpark transforms.
3. Deploy: databricks bundle deploy -t staging|prod from CI, promoting the same artifact.
4. Test on real data: run the job against a staging catalog, then compare row counts and business metrics.
5. Guardrails: Unity Catalog grants, job permissions, and cluster policies are code, reviewed like application code.
```

Key practices: keep transformation logic in importable Python modules under test, parameterize catalogs per environment, and never point CI at the production catalog.

Continuous integration for Databricks rests on treating notebooks and job definitions as code. Transformations live in importable Python modules so they can be imported and tested without a Spark cluster where possible, notebooks are thin wrappers that call those modules, and job and pipeline definitions live in a bundle in the same repository. A pull request runs `databricks bundle validate` plus the unit tests, and a merge deploys to a staging target with the same artifact that will later go to production. Authentication in CI uses a service principal or workload identity, never a personal access token belonging to an engineer.

The testing ladder is worth naming explicitly in an interview. At the bottom, pure transformations are tested locally with `pytest` on small DataFrames. In the middle, integration tests run the job against a staging catalog that mirrors production structure but holds sample data. At the top, post-deploy checks compare row counts, null rates, and a few business metrics against expectations, and alert when they drift. Data-quality expectations inside a pipeline or checks written as SQL both work; the important part is that the check runs on every deploy rather than in a quarterly audit.

The mistake to avoid is pointing CI at the production catalog because it is the only one with data. That couples a test failure to live tables and makes a bad deploy capable of corrupting the thing you were protecting. The second is testing only by clicking Run in a notebook, which proves the code ran once on one person's cluster and nothing about whether it is reproducible. The third is storing cloud credentials in CI variables long after the job moved to Unity Catalog external locations.

## What security and access-mode concepts matter?

+ **Access modes** - dedicated (formerly single-user), standard (formerly shared), and serverless; the mode determines which Spark features and languages are available and how users are isolated.
+ **Identity** - users, groups, service principals, and managed identities for jobs; avoid personal access tokens in production.
+ **Secrets** - Databricks secret scopes backed by Key Vault/Secrets Manager; never put credentials in notebooks or cluster env vars.
+ **Network** - private link/VNet injection, IP access lists, and no-public-IP clusters for regulated environments.
+ **Audit** - `system.access.audit` records who read, wrote, or changed what.

Access mode decides who may share compute and how users are isolated. **Standard** mode is the default for modern runtimes and allows multiple users on one cluster with data isolation between them; it supports Python, SQL, and Scala. **Dedicated** mode assigns the cluster to a single user or group and adds R support, at the cost of sharing. Selection is usually automatic, and it defaults to Standard unless an ML runtime, a GPU instance type, or an older runtime forces Dedicated. Tables with row filters and column masks need Standard mode on a supported runtime, or they fall back to serverless enforcement on Dedicated compute, which has its own limitations.

Identity and secrets have their own mechanics. Interactive users authenticate as themselves; jobs should run as a service principal or a workload identity so the pipeline keeps working when a person leaves. Secret scopes backed by Key Vault or Secrets Manager hold credentials, and `{{secrets/scope/key}}` injects them into Spark configuration or environment variables without ever printing them. Network controls such as private link, IP access lists, and no-public-IP clusters determine whether compute can reach storage and users can reach the workspace, and `system.access.audit` records who read, wrote, or changed what.

The recurring mistake is a long-lived personal access token wired into a production job, which means the job's permissions are one person's permissions and revoking that person breaks production. The second is putting a database password in a notebook or a cluster environment variable, where it is visible to anyone who can view the cluster configuration. The third is assuming Unity Catalog alone is security: it governs data access, not who can attach to a cluster or exfiltrate a result, so cluster policies and audit review are part of the same story.

## How do you control Databricks cost?

```sql
-- top spend by SKU and workspace, from the billing system table
SELECT usage_date, sku_name, SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY usage_date, sku_name
ORDER BY dbus DESC;
```

Levers that actually move the bill:

+ Job clusters with auto-termination instead of idle all-purpose clusters.
+ Spot instances with fallback for batch, plus autoscaling ranges that reflect real need.
+ Photon and liquid clustering to finish jobs sooner (DBUs are time-based).
+ Right-sized SQL warehouses and auto-stop; separate warehouses per workload instead of one giant one.
+ Serverless for bursty jobs where cluster startup would otherwise dominate.
+ Tag clusters/jobs by team and cost center, then review the top 20 spenders monthly.

Databricks cost has four parts and they are billed by different parties. Databricks bills **DBUs**, the software units of compute time, at a SKU rate that depends on the product (jobs, all-purpose, SQL, pipelines) and whether Photon and serverless are used. Your cloud provider bills the **virtual machines and disks** that classic compute runs on, plus **storage** and any **egress**. Serverless folds the VM cost into a higher DBU rate. So the arithmetic for a classic job run is `node count x DBUs per node-hour x runtime hours x price per DBU`, plus the cloud VM bill for the same wall-clock time, which is why a cheaper rate and a shorter runtime both show up in the same number.

Use the billing system tables rather than a spreadsheet. `system.billing.usage` holds one row per usage record with `usage_quantity` in `usage_unit` (usually `DBU`), and it nests the resource identifiers: `usage_metadata.cluster_id` for classic compute, `usage_metadata.job_id` and `usage_metadata.job_run_id` for jobs, `usage_metadata.warehouse_id` for SQL warehouses, and `usage_metadata.dlt_pipeline_id` for pipelines. `usage_metadata` does not populate `job_id` for jobs that ran on all-purpose compute, so those runs are attributed by `cluster_id` instead. Pricing lives in `system.billing.list_prices`, where the `pricing` struct exposes `default` and `effective_list`. Join the two to turn DBUs into currency.

Then apply the levers in order of size. Move scheduled work from all-purpose to job compute, which changes the SKU rate rather than the runtime. Use spot instances with fallback for batch and keep the driver on-demand. Right-size autoscaling ranges and turn on auto-stop everywhere. Enable Photon and liquid clustering so jobs finish sooner, remembering that DBUs are time-based. Separate SQL warehouses by workload with short auto-stop. Tag clusters and jobs by team and cost center so the monthly review has an owner for every line, and remember that an idle cluster is the single most common source of surprise spend.

## How does Databricks compare with Snowflake and BigQuery?

| Aspect | Databricks | Snowflake | BigQuery |
|---|---|---|---|
| Engine | Spark + Photon, plus SQL warehouses | proprietary SQL engine | serverless SQL engine |
| Table format | Delta (and Iceberg) on your storage | Snowflake-managed micro-partitions (Iceberg tables available) | BigQuery-managed storage (BigLake/Iceberg available) |
| Strength | large-scale Spark/ML, notebooks, streaming, open formats | SQL analytics, ease of use, governed sharing | serverless simplicity, tight GCP integration |
| Cost model | DBUs + cloud VM/storage | credits + storage | on-demand/edition slots + storage |
| Governance | Unity Catalog | native RBAC + policies | IAM + Dataplex/policy tags |

The honest interview answer: choose Databricks when Spark-scale engineering, streaming, ML, and open table formats matter; choose Snowflake or BigQuery when the workload is mostly SQL analytics and managed simplicity wins. Many companies run two of the three.

The comparison is easier once you separate the engine from the billing model. Databricks runs Spark plus Photon over Delta or Iceberg files in your storage, and charges DBUs per compute type on top of your cloud VM bill. Snowflake runs a proprietary SQL engine over its own micro-partitioned storage and charges credits per warehouse-second plus storage. BigQuery runs a serverless engine over Google-managed storage with on-demand or slot-based pricing. All three can read open formats now, so the differentiator is usually where the data lives, who governs it, and what else has to run against it.

Pick by workload shape. Databricks wins when the work is large-scale Spark engineering, streaming, machine learning, or a lakehouse where one governed copy of data serves BI, data science, and pipelines, and when open table formats and avoiding lock-in matter. Snowflake wins when the job is mostly SQL analytics, governed sharing, and low operational overhead for analysts. BigQuery wins when the estate is already on GCP and the priority is serverless simplicity. Plenty of companies run two of the three, with one as the lakehouse and one as the SQL-serving layer, and saying so is a better answer than declaring a winner.

The traps in this comparison are overconfident claims. Spark is not always slower than a dedicated warehouse, and a Photon job over liquid-clustered Delta is competitive on the workloads Databricks targets. Snowflake is not automatically easier, because a badly clustered table and an undersized warehouse are just as slow there. And cost per terabyte scanned is not cost per useful answer: the metric that decides is cost per pipeline run at the required latency, which depends on file layout, cluster sizing, and caching in every one of the three.

## How do you migrate from on-prem Spark and Hive?

+ **Storage first** - copy HDFS/Hive data to object storage (DistCp, cloud transfer service), then register it as external Delta/Iceberg tables or migrate in place.
+ **Metastore** - move Hive metastore objects into Unity Catalog; use the migration tooling to upgrade tables to Delta and fix unsupported types.
+ **Code** - replace `hive`-specific SQL with Spark SQL equivalents, remove `HiveContext`, and convert Python UDFs to built-in functions.
+ **Jobs** - port Oozie/Airflow DAGs to Databricks Workflows or keep Airflow and call Databricks jobs via the operator.
+ **Validate** - compare row counts and checksums between old and new tables, then run both paths in parallel for a period before cutover.

Migrate in a fixed order, because each step depends on the previous one. Move the bytes first, from HDFS or a Hive warehouse onto object storage, using DistCp or a cloud transfer service, and land them in a raw zone. Then migrate the catalog: register the data in Unity Catalog, either as external tables in place or by converting to managed Delta tables, and repair the types and partitioning that Hive tolerated but Delta does not. Only then port the code, then the schedules, and finally validate. Doing the metastore before the storage leaves tables pointing at paths that no longer exist.

The conversion details decide how painful the middle is. Hive types that Delta does not accept need mapping, and `HiveContext` and `hive`-specific SQL need replacing with Spark SQL. Python UDFs are usually the worst offenders for performance, because Photon does not accelerate them; most can be rewritten as built-in functions or expressions. Oozie and cron schedules become Databricks Workflows, or stay in Airflow and call Databricks jobs through the operator. During cutover, run the old and new pipelines in parallel and compare row counts, checksums, and a handful of business metrics before switching consumers over.

The mistake that turns a migration into an incident is a big-bang cutover with no parallel period, which leaves no way to tell whether a discrepancy is a bug or a definition change. The second is assuming Hive SQL runs unchanged: implicit casts, non-deterministic ordering, and `INSERT OVERWRITE` semantics all differ, and a query that passed in Hive can quietly produce different results. The third is migrating the tables but not the access model, so the old Hive grants disappear and half the analysts lose access on day one.

## What is MLflow and how is it used on Databricks?

MLflow tracks experiments, packages models, and manages a model registry. On Databricks it is integrated with the workspace and Unity Catalog.

In [ ]:
# ILLUSTRATIVE - requires mlflow: install to run, or read on.
import mlflow
import mlflow.sklearn

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="churn-baseline"):
    mlflow.log_params({"max_depth": 8, "n_estimators": 200})
    mlflow.log_metric("auc", 0.912)
    mlflow.sklearn.log_model(model, artifact_path="model",
                             registered_model_name="acme.ml.churn_classifier")

Data-engineer relevance: feature tables built with Spark feed training jobs; the same features should be served online from a low-latency store, which is why feature consistency and point-in-time correctness come up in interviews.

MLflow has four moving parts. **Tracking** records experiments and runs, with parameters, metrics, tags, and artifacts, so a result can be reproduced. **Projects** package code and environment, though on Databricks the notebook or wheel usually plays that role. **Models** package a trained model with its signature and dependencies, in the `mlflow.sklearn`-style flavor that knows how to load it. The **model registry** versions models and assigns aliases; on Databricks the registry is Unity Catalog, addressed as `catalog.schema.model_name`, which is why `mlflow.set_registry_uri("databricks-uc")` appears in code that registers a model.

For a data engineer the interesting parts are the data contracts around a model. Feature tables built with Spark feed training jobs, and the same features must be served at low latency for inference, which is why feature consistency and point-in-time correctness dominate the discussion. Point-in-time correctness means a training row must use the feature values that were actually known at the label's timestamp, not the latest values, or the model learns from the future. Lineage from table to feature to model to endpoint is what makes an audit or a rollback possible, and Unity Catalog provides it end to end.

The mistakes are small and expensive. Logging the entire training dataset as an artifact bloats the run store when a Delta table reference would do. Registering a model without a signature or an input example means the serving layer cannot validate the request. Writing artifacts to a notebook-local path makes the run unreproducible as soon as the cluster restarts. And skipping a baseline run means there is no reference point to compare a new model against, so a regression is invisible until it reaches production.

## What are common Databricks interview traps?

+ Saying "Databricks is just hosted Spark" - it also includes Delta, Unity Catalog, workflows, SQL warehouses, and governance.
+ Running production pipelines on an all-purpose cluster.
+ Using `collect()` or `toPandas()` on large data and blaming the cluster.
+ Calling Python UDFs on huge datasets and expecting Photon to fix it.
+ Over-partitioning a Delta table, then wondering why the file listing is slow.
+ Confusing `VACUUM` retention with time travel: vacuuming inside the retention window breaks time travel.
+ Forgetting that Unity Catalog changes the namespace to three levels, so old two-level paths stop working.
+ Assuming autoscaling fixes skew - it cannot split a single hot partition.

Answer these as two-part statements: name the mechanism, then name when it breaks. For example, "autoscaling adds workers based on pending tasks, so it helps a widening job but cannot split a single hot partition" is a much stronger answer than either half alone. Interviewers are listening for whether you know where a feature stops working, because that is what separates someone who has operated the platform from someone who has read the marketing page.

A few more traps worth having ready. Assuming Unity Catalog is optional overhead rather than the namespace and grant layer every job uses. Believing that `OPTIMIZE` is free, when it rewrites files and consumes DBUs. Treating `VACUUM` as routine cleanup when it silently ends the time-travel window and deletes Change Data Feed history. Scheduling `OPTIMIZE` on top of predictive optimization so the same table is maintained twice. Forgetting that serverless compute still needs network paths to your storage. And measuring a performance change with wall-clock time alone rather than DBUs per run and the query profile.

Before proposing any change, be able to say which of three problems you are solving: the query reads too much data, the query does too much work per row, or the workload is waiting for compute. Data layout and pruning address the first, Photon and rewritten logic address the second, and sizing, autoscaling, and workload isolation address the third. A change that does not map to one of those three is usually a guess, and the cost tables will eventually show it.

## How do you tune Databricks jobs for performance?

Tuning on Databricks is a diagnosis, not a hunt for a magic setting. Start by classifying the problem into one of three buckets: the job reads too much data, the job does too much work per row, or the job is waiting for compute that is not there. The first bucket is a layout and pruning problem, the second is a query-shape and engine problem, and the third is a sizing and isolation problem. Naming the bucket first prevents the classic mistake of adding workers to a job whose real issue is a full scan of a poorly clustered table.

The evidence comes from three places. The **query profile** in Databricks SQL and serverless shows per-operator time, rows, bytes, spill, and how much of the work ran in Photon. The **Spark UI** on classic compute shows stage and task distributions, which is where skew becomes visible as a few tasks taking far longer than the rest. The **system tables** give the historical view: `system.query.history` records per-statement duration, bytes read, shuffle, spill, and cache hits, and `system.billing.usage` turns the result into DBUs so you can see whether a change saved money or only saved wall-clock time.

Work the levers in this order, because each one is cheaper than the next. First reduce the data: filter earlier, select fewer columns, add partition or clustering keys, and compact small files. Second reduce the work per row: replace Python UDFs with built-in functions, avoid `collect()` and `toPandas()` on large results, and let Photon cover the heavy operators. Third change the compute: right-size workers, enable autoscaling, isolate the workload, and choose job or serverless compute. Only after those should you consider a bigger cluster, and by then you should be able to say which operator and which metric justified it.

```sql
-- The historical view: which statements actually spent the time, and why.
SELECT statement_id,
       executed_by,
       compute.type                AS compute_type,
       compute.warehouse_id        AS warehouse_id,
       statement_type,
       total_duration_ms,
       execution_duration_ms,
       compilation_duration_ms,
       waiting_at_capacity_duration_ms,
       read_bytes,
       shuffle_read_bytes,
       spilled_local_bytes,
       from_result_cache
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 1 DAY
  AND execution_status = 'FINISHED'
ORDER BY execution_duration_ms DESC
LIMIT 25;

-- The same data rolled up by compute so you can see which resource is busy.
SELECT compute.type          AS compute_type,
       compute.warehouse_id  AS warehouse_id,
       COUNT(*)              AS statements,
       SUM(execution_duration_ms) / 1000        AS execution_seconds,
       SUM(waiting_at_capacity_duration_ms) / 1000 AS queued_seconds,
       SUM(read_bytes) / POWER(1024, 3)         AS read_gb,
       SUM(shuffle_read_bytes) / POWER(1024, 3) AS shuffle_gb,
       SUM(spilled_local_bytes) / POWER(1024, 3) AS spilled_gb
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 7 DAYS
GROUP BY ALL
ORDER BY execution_seconds DESC;
```


Read the three durations together. If `waiting_at_capacity_duration_ms` dominates, the query is queued and the fix is concurrency or warehouse sizing, not query rewriting. If `execution_duration_ms` dominates and `spilled_local_bytes` is large, the query ran out of memory on its operators and the fix is to reduce what it holds in memory or give it more. If `read_bytes` is close to the full table size and `from_result_cache` is false, the query is scanning and the fix is layout, pruning, or caching. One metric, one diagnosis, one change, then re-run and compare the same columns.

## How do you lay out Delta tables for fast queries?

Liquid clustering is the recommended layout for new Delta tables. You declare the columns most often used in filters with `CLUSTER BY`, and the engine organizes file ranges so that a predicate can skip almost everything else. It is incremental: `OPTIMIZE` rewrites only the data that needs reclustering, not the whole table. Keys can be changed later with `ALTER TABLE ... CLUSTER BY`, and the new keys apply to future writes and optimizations without rewriting all history. `CLUSTER BY AUTO` goes further and lets predictive optimization choose keys from the observed query workload, which is the right default for a table whose access pattern is not yet known.

Partitioning is not obsolete, but it is narrower than it used to be. Partitioning is right when a genuinely low-cardinality column is filtered by almost every query, such as a `region` or a coarse `event_date`, and the table is large enough that directory-level pruning matters. It is wrong for high-cardinality columns, where it creates millions of tiny files and a metadata problem that costs more than the pruning saves. **ZORDER** co-locates rows by a set of columns within a file, and it still works on partitioned or unpartitioned tables, but it is incompatible with liquid clustering and is no longer the recommended starting point.

Choose by the query pattern. If filters are on one or two high-cardinality columns and the pattern is stable, use `CLUSTER BY` with those columns and run `OPTIMIZE` regularly; up to four clustering keys are allowed, and fewer keys often perform better on smaller tables. If one very low-cardinality column appears in every predicate, partition by it and stop there. If the workload is unknown or changing, `CLUSTER BY AUTO` is the low-risk choice. Change clustering keys only when the old keys stopped matching the queries, and remember that existing data keeps its old layout until `OPTIMIZE ... FULL` reclusters it.

```sql
-- Liquid clustering from the start, and the incremental maintenance that powers it.
CREATE OR REPLACE TABLE acme.gold.fct_orders (
  order_id     STRING NOT NULL,
  customer_id  STRING,
  order_date   DATE,
  amount       DECIMAL(18, 2)
)
USING DELTA
CLUSTER BY (order_date, customer_id);

INSERT INTO acme.gold.fct_orders SELECT * FROM acme.silver.orders_stage;

-- Incremental: rewrites only what needs to move.
OPTIMIZE acme.gold.fct_orders;

-- After changing keys, recluster all existing data once.
ALTER TABLE acme.gold.fct_orders CLUSTER BY (customer_id, order_date);
OPTIMIZE acme.gold.fct_orders FULL;

-- Let the platform pick keys from the real workload.
ALTER TABLE acme.gold.fct_orders CLUSTER BY AUTO;

-- Inspect what the table currently uses.
DESCRIBE DETAIL acme.gold.fct_orders;

-- Partitioning is the alternative, never the companion.
CREATE OR REPLACE TABLE acme.gold.fct_orders_partitioned (
  order_id STRING, customer_id STRING, order_date DATE, amount DECIMAL(18, 2)
)
USING DELTA
PARTITIONED BY (order_date);
```


Target file size is the second half of layout. Aim for roughly 100 MB to 1 GB per file so that a scan has enough files to parallelize across and few enough to avoid metadata overhead. Many small files are the usual failure: streaming appends create thousands of tiny files unless auto compaction or a scheduled `OPTIMIZE` merges them, and a query then spends more time listing than reading. A few enormous files are the opposite failure: they cap parallelism and make every rewrite expensive. `OPTIMIZE` with a target size and a regular schedule, or auto compaction for streaming tables, keeps the sizes in range.

The mistakes to name in an interview are over-partitioning, combining techniques that do not combine, and expecting a key change to take effect for free. Partitioning by `customer_id` on a billion-row table produces a directory per customer and destroys performance. Liquid clustering and partitioning cannot coexist on one table, and neither can clustering and `ZORDER`. Changing `CLUSTER BY` on a large table without a subsequent `OPTIMIZE ... FULL` leaves old data clustered on old keys, so the measurement looks like clustering simply does not work.

## How do you keep Delta tables healthy over time?

Delta maintenance has three operations with three different jobs. **`OPTIMIZE`** compacts small files into larger ones and, on a clustered table, incrementally clusters the new data. **`VACUUM`** deletes data files that are no longer referenced by the current table version and that are older than the retention threshold. **`ANALYZE`** collects statistics that the optimizer uses for join ordering and size estimation. Compaction and clustering improve read performance, vacuum reduces storage cost and permanently removes deleted records, and analyze improves plan quality. All three consume compute, so each is a cost to weigh against its benefit.

Retention is where vacuum and time travel collide. Data files are kept for `delta.deletedFileRetentionDuration`, which defaults to 7 days, while the transaction log is kept for about 30 days. Running `VACUUM` with the default threshold therefore removes the ability to time travel to versions older than a week, and it also removes Change Data Feed history for that window. `VACUUM table RETAIN 168 HOURS DRY RUN` previews what would be deleted without deleting it, which is the safe way to check before a larger retention change. Set the retention deliberately, before enabling anything that vacuums automatically.

Predictive optimization runs `OPTIMIZE`, `VACUUM`, and `ANALYZE` automatically for Unity Catalog managed tables, and it is enabled by default on newer accounts. It bills as serverless jobs compute, it respects `delta.deletedFileRetentionDuration`, and it can also select liquid clustering keys. The important consequence is operational: if predictive optimization is enabled, a hand-written hourly `OPTIMIZE` job is usually redundant work and can run concurrently with the automatic one. Let the platform handle maintenance on managed tables, and keep manual jobs for external tables and for workloads that need a specific schedule.

```sql
-- Compaction and incremental clustering.
OPTIMIZE acme.gold.fct_orders;
OPTIMIZE acme.gold.fct_orders WHERE order_date >= '2026-01-01';

-- Rebuild the layout after a key change (expensive on large tables).
OPTIMIZE acme.gold.fct_orders FULL;

-- Statistics for the optimizer.
ANALYZE TABLE acme.gold.fct_orders COMPUTE STATISTICS FOR ALL COLUMNS;

-- VACUUM: preview first, then delete, and know what you are giving up.
VACUUM acme.gold.fct_orders RETAIN 168 HOURS DRY RUN;
VACUUM acme.gold.fct_orders RETAIN 168 HOURS;

-- Keep a longer time-travel window before enabling automatic vacuuming.
ALTER TABLE acme.gold.fct_orders
  SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days');

-- Hand maintenance back to the platform for managed tables.
ALTER TABLE acme.gold.fct_orders ENABLE PREDICTIVE OPTIMIZATION;

-- Apply soft deletes so old values leave the current files.
REORG TABLE acme.gold.fct_orders APPLY (PURGE);
```


The cost of each operation is worth stating plainly. `OPTIMIZE` reads and rewrites files, so it costs DBUs proportional to the data it moves; on a clustered table the incremental version is cheap, while `OPTIMIZE ... FULL` on a large table can run for hours. `VACUUM` lists the table directory and then deletes files from the driver, so its cost is mostly listing and driver time; a cluster with one to four workers and a driver of 8 to 32 cores is the documented sweet spot. `ANALYZE` scans the table to build statistics and is usually cheap relative to the queries it improves. Predictive optimization runs on serverless jobs compute, so its cost shows up in the billing tables under the serverless jobs SKU rather than on your own cluster.

The mistakes are vacuuming inside the time-travel window you still need, scheduling `OPTIMIZE` on top of predictive optimization, and treating maintenance as free. A team that runs `VACUUM` nightly with a one-day retention and then tries to recover from a bad write discovers that the recovery point no longer exists. A team that enables predictive optimization and keeps its old `OPTIMIZE` job pays twice for overlapping work. And a team that never compacts a streaming table ends up with millions of small files and a scan that reads a few kilobytes per file, which is pure overhead.

## How do you tune joins, shuffles and skew on Databricks?

Adaptive query execution, or AQE, is the first thing to understand because it is enabled by default and it changes the plan while the query runs. It coalesces many small post-shuffle partitions into fewer, better-sized ones, so the old advice about hand-tuning `spark.sql.shuffle.partitions` to match your data is mostly obsolete; leave it at the default and let the coalescing work. It can switch a sort-merge join to a broadcast join when one side turns out to be small after filtering, and it detects and splits skewed partitions on a join when they exceed both a size threshold and a multiple of the median.

The knobs that matter are few. `spark.sql.adaptive.coalescePartitions.enabled` controls partition coalescing, `spark.sql.adaptive.advisoryPartitionSizeInBytes` sets the target post-shuffle partition size, and `spark.sql.adaptive.skewJoin.enabled` plus `spark.sql.adaptive.skewJoin.skewedPartitionFactor` and `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` control skew handling. The factor is how many times larger than the median a partition must be before it is treated as skewed, and the threshold is the absolute size floor. Raising the factor makes the engine less aggressive about splitting, which is sometimes what you want when the split overhead exceeds the gain.

Broadcast and skew hints are scalpel tools for the cases AQE cannot fix on its own. `/*+ BROADCAST(dim_table) */` forces a broadcast join regardless of the automatic threshold, which is right when you know a dimension is small but statistics are missing or stale. The `SKEW` hint names the relation and, optionally, the join columns that are skewed, so the engine builds a better plan for exactly those joins; specifying the skewed values themselves reduces the detection overhead further. Salting is the manual fallback when AQE and hints still leave one task far behind: add a random bucket to the hot key on both sides so the hot value spreads across many tasks.

```sql
-- Let AQE do the standard work; tune only when the profile says you must.
SET spark.sql.adaptive.enabled = true;
SET spark.sql.adaptive.coalescePartitions.enabled = true;
SET spark.sql.adaptive.advisoryPartitionSizeInBytes = 128m;
SET spark.sql.adaptive.skewJoin.enabled = true;
SET spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5;
SET spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 256m;

-- Broadcast a small dimension when statistics are unreliable.
SELECT /*+ BROADCAST(d) */
       f.order_id, f.amount, d.segment
FROM acme.gold.fct_orders f
JOIN acme.gold.dim_customer d ON d.customer_id = f.customer_id;

-- Tell the engine which relation and columns are skewed.
SELECT /*+ SKEW('f', 'customer_id') */
       f.customer_id, COUNT(*) AS orders
FROM acme.gold.fct_orders f
JOIN acme.gold.dim_customer d ON d.customer_id = f.customer_id
GROUP BY f.customer_id;
```


In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# Manual salting: the fallback when AQE and hints still leave a straggler task.
from pyspark.sql import functions as F

SALT_BUCKETS = 32

orders = spark.read.table("acme.gold.fct_orders")
customers = spark.read.table("acme.gold.dim_customer")

salted_orders = (
    orders.withColumn("salt", (F.rand(seed=42) * SALT_BUCKETS).cast("int"))
    .withColumn("join_key", F.concat_ws("#", "customer_id", "salt"))
)

exploded_customers = (
    customers.select("customer_id", "segment")
    .withColumn("salt", F.explode(F.array(*[F.lit(i) for i in range(SALT_BUCKETS)])))
    .withColumn("join_key", F.concat_ws("#", "customer_id", "salt"))
)

joined = salted_orders.join(exploded_customers, on="join_key").drop("salt", "join_key")


Read the query profile or the Spark UI before reaching for any of these. In the profile, a join operator with one task far larger than the rest is skew; a join with a huge shuffle on both sides may want a broadcast; an operator with heavy spill wants less data before it, not a different join strategy. In the Spark UI, sort the stage's tasks by duration and look at the first few: one task at ten times the median is skew, while a uniform but slow stage is a partitioning or sizing problem. `spark.sql.shuffle.partitions` staying at its default is correct under AQE; changing it manually is a hint that you have not looked at the profile yet.

The mistakes are treating broadcast as always better, salting before checking whether AQE already solved it, and confusing a shuffle problem with a skew problem. A broadcast join of a table that is not actually small fails or spills on the driver side. Salting multiplies the small side by the bucket count, so applying it to a join that was not skewed makes the job slower. And a stage that is uniformly slow because it has too few partitions needs coalescing or a different key, not a skew workaround.

## How do you size and isolate compute for cost and speed?

Compute choice is a three-way decision between type, size, and isolation. The type sets the billing rate: jobs compute is cheaper per DBU than all-purpose, serverless folds the cloud cost in and removes cluster management, and SQL warehouses are priced for SQL concurrency. The size sets how fast one unit of work finishes and how much it costs per hour. Isolation decides who shares a resource and therefore who can affect whom. A good default for a production pipeline is a job cluster or serverless job, Photon enabled, an autoscaling range that reflects the largest stage, spot with on-demand fallback for batch, and a driver sized for the results the job actually collects.

Autoscaling ranges should come from measurement, not optimism. Look at the number of tasks in the widest stage, the slots per worker (roughly one task per core), and the target stage duration, and set the maximum to the workers that number implies plus some headroom. Set the minimum above one when the job always needs parallelism at the start, since scaling up costs startup time. Bear in mind that the range does not fix skew, that scaling down on streaming workloads is limited, and that Spot instances can be reclaimed, so a stateful streaming driver should stay on-demand.

Cluster policies are the mechanism that makes sizing consistent across a team. A policy fixes or constrains fields such as node types, maximum workers, Photon, auto-termination, and tags, so an engineer cannot accidentally create a 200-node cluster for a test. Combine policies with tags so every DBU in `system.billing.usage` carries a team and a cost center, and separate workloads onto separate compute so a heavy backfill cannot starve a dashboard. Remember the DBU-versus-cloud split when you compare options: a serverless run looks more expensive per DBU because it includes the infrastructure, so compare total cost, not the unit rate.

```json
{
  "cluster_name": "etl-daily",
  "policy_id": "0011AABB22CC33DD",
  "spark_version": "15.4.x-scala2.12",
  "node_type_id": "m5d.4xlarge",
  "driver_node_type_id": "m5d.2xlarge",
  "autoscale": {"min_workers": 4, "max_workers": 32},
  "aws_attributes": {
    "availability": "SPOT_WITH_FALLBACK",
    "zone_id": "auto",
    "first_on_demand": 1
  },
  "runtime_engine": "PHOTON",
  "data_security_mode": "USER_ISOLATION",
  "autotermination_minutes": 20,
  "custom_tags": {"team": "data-platform", "cost-center": "analytics"},
  "spark_conf": {"spark.sql.adaptive.enabled": "true"}
}
```


```sql
-- SQL warehouses scale by size and by cluster count, and stop when idle.
CREATE WAREHOUSE IF NOT EXISTS bi_wh
  WITH WAREHOUSE_SIZE = 'SMALL'
       MIN_CLUSTER_COUNT = 1
       MAX_CLUSTER_COUNT = 5
       SCALING_POLICY = 'STANDARD'
       AUTO_STOP_MINS = 10
       ENABLE_SERVERLESS_COMPUTE = true;

-- A separate warehouse for a heavy extract, so the dashboard is unaffected.
CREATE WAREHOUSE IF NOT EXISTS extract_wh
  WITH WAREHOUSE_SIZE = 'LARGE'
       MIN_CLUSTER_COUNT = 1
       MAX_CLUSTER_COUNT = 2
       AUTO_STOP_MINS = 5
       ENABLE_SERVERLESS_COMPUTE = true;
```


The mistakes to avoid are one shared cluster for everything, autoscaling ranges copied from a blog post, and comparing unit prices instead of totals. Sharing one all-purpose cluster between interactive work and a scheduled pipeline couples their failures and keeps the premium rate running for the notebook user. Copying a range without measuring produces a maximum that is never reached, which is harmless, or a minimum that is too low, which adds startup cost to every run. And choosing all-purpose over jobs compute because the DBU rate looked similar ignores that the all-purpose cluster also idles and bills.

## How do you measure Databricks performance and cost?

Measurement has four layers, and they answer different questions. The **query profile** answers why one statement was slow, operator by operator, with rows, bytes, spill, and Photon coverage. The **Spark UI** answers where a job's stages spent time and whether task durations were uniform. **`system.query.history`** answers which statements across the account were expensive over time, with the compute resource nested under `compute`. **`system.billing.usage`** answers what the DBUs cost, with resource identifiers nested under `usage_metadata` and list prices in `system.billing.list_prices`. Use the profile for a single query, the system tables for a trend or a budget.

`system.billing.usage` needs one structural fact to be read correctly: the interesting identifiers are inside `usage_metadata`, not at the top level. `usage_metadata.cluster_id` is populated for classic compute, `usage_metadata.job_id` and `usage_metadata.job_run_id` for jobs on job or serverless compute, `usage_metadata.warehouse_id` for SQL warehouses, and `usage_metadata.dlt_pipeline_id` for Lakeflow pipelines. A job that ran on an all-purpose cluster has no `job_id` in the usage record, so it must be attributed by `cluster_id`. `usage_quantity` carries the units named in `usage_unit`, which is `DBU` for compute. Unknown or changed fields are best handled with `DESCRIBE system.billing.usage` rather than guessed.

Cost is a join, not a column. `system.billing.usage` records DBUs; `system.billing.list_prices` records a historical log of SKU prices, where `pricing.default` is the published list price and `pricing.effective_list` resolves list and promotional prices. Join on `sku_name` and a time condition, because the price table only has rows when a price changed, so you need the most recent price at or before the usage date. Filter to `record_type = 'ORIGINAL'` when you want a clean total, or aggregate and let retraction rows cancel the originals, as the billing reference describes.

```sql
-- DBUs by SKU and day, from the usage table alone.
SELECT usage_date,
       sku_name,
       billing_origin_product,
       product_features.is_photon AS is_photon,
       SUM(usage_quantity)       AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY ALL
ORDER BY usage_date DESC, dbus DESC;

-- Attribute spend to jobs and clusters; note the nesting under usage_metadata.
SELECT usage_date,
       usage_metadata.cluster_id   AS cluster_id,
       usage_metadata.job_id       AS job_id,
       usage_metadata.warehouse_id AS warehouse_id,
       usage_metadata.dlt_pipeline_id AS pipeline_id,
       SUM(usage_quantity)         AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY ALL
ORDER BY dbus DESC
LIMIT 20;

-- Turn DBUs into currency with the historical price log.
SELECT u.usage_date,
       u.sku_name,
       SUM(u.usage_quantity) AS dbus,
       SUM(u.usage_quantity * CAST(p.pricing.effective_list.default AS DECIMAL(18, 6))) AS list_cost
FROM system.billing.usage u
LEFT JOIN system.billing.list_prices p
  ON p.sku_name = u.sku_name
 AND p.price_start_time <= u.usage_date
 AND (p.price_end_time IS NULL OR p.price_end_time > u.usage_date)
WHERE u.usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY ALL
ORDER BY list_cost DESC;

-- Attribute by the tags that were set on the cluster or job.
SELECT custom_tags['team']        AS team,
       custom_tags['cost-center'] AS cost_center,
       SUM(usage_quantity)        AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY ALL
ORDER BY dbus DESC;
```


`system.query.history` is the performance counterpart and has its own nesting: the warehouse or compute type lives under the `compute` struct, as `compute.type` (`WAREHOUSE`, `SERVERLESS_COMPUTE`, or `CLASSIC_COMPUTE`) and `compute.warehouse_id`. Alongside the duration columns it carries scan and shuffle metrics: `read_bytes`, `read_files`, `pruned_files`, `read_rows`, `produced_rows`, `written_bytes`, `shuffle_read_bytes`, and `spilled_local_bytes`. `from_result_cache` tells you a query was served without executing, and `query_tags` carries the key-value tags set on the session, which is how you isolate one tuning experiment from everything else that ran.

```sql
-- The tagged before-and-after: same tags, compare the columns that matter.
SELECT query_tags['experiment'] AS experiment,
       COUNT(*)                 AS statements,
       SUM(execution_duration_ms) / 1000 AS execution_seconds,
       SUM(read_bytes) / POWER(1024, 3)  AS read_gb,
       SUM(shuffle_read_bytes) / POWER(1024, 3) AS shuffle_gb,
       SUM(spilled_local_bytes) / POWER(1024, 3) AS spilled_gb,
       SUM(CASE WHEN from_result_cache THEN 1 ELSE 0 END) AS cached_statements
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 1 DAY
  AND query_tags['experiment'] IS NOT NULL
GROUP BY ALL
ORDER BY experiment;
```


A before-and-after checklist keeps the comparison honest. Record the baseline first: DBUs per run from `system.billing.usage`, the p50 and p95 duration from `system.query.history`, bytes read, shuffle bytes, and spill. Change exactly one thing, such as a clustering key or the Photon setting, and tag the run. Re-run the same workload on comparable data, not on a smaller sample. Compare the same columns, and accept the change only if it improves the metric you set out to improve without regressing another. If the wall-clock time fell but the DBUs rose by the same proportion, you bought latency with money, which may or may not be the trade you wanted.

The mistakes here are measuring without a baseline, measuring on different data, and trusting a single number. A wall-clock improvement on a warm cache is not a layout improvement. A DBU reduction that comes from a shorter schedule is not an efficiency gain. And an aggregate that mixes interactive and scheduled work hides the workload you were trying to fix, which is why the nesting under `usage_metadata` and `compute` and the tags on clusters and queries exist in the first place.

## Databricks exercises

These exercises use only the Python standard library and model the sizing, cost, and layout arithmetic behind the sections above. Each one states the task, gives a hint, names the expected result, and then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are illustrative, not a price list. The point of each exercise is to know which quantity to compute and which direction a change moves it: DBUs scale with node count and runtime, Photon costs more per hour but can cost less per job, a hot partition is fixed by spreading the key rather than by adding workers, and file count follows directly from table size and target file size. Treat the asserts as a check on the formula.

### Try it 1. Pick a node family from the workload shape

**Task.** For each workload, choose a node family. Prefer storage-optimized when the job spills more than 20 percent of its input to local disk, memory-optimized when it shuffles at least twice its input size, and compute-optimized otherwise. The metric is `spill_gb / data_gb`.

**Hint.** Check the spill ratio first, because disk pressure dominates; then the shuffle ratio.

**Expected result.** `nightly_etl` is memory-optimized, `feature_math` is compute-optimized, and `log_parse` is storage-optimized.

In [ ]:
def pick_node_family(workload):
    if workload["spill_gb"] / workload["data_gb"] > 0.2:
        return "storage-optimized (spills to local disk)"
    if workload["shuffle_ratio"] >= 2:
        return "memory-optimized (large shuffles and caches)"
    return "compute-optimized (CPU-bound, little shuffle)"


workloads = {
    "nightly_etl": {"data_gb": 800, "shuffle_ratio": 4.0, "spill_gb": 120},
    "feature_math": {"data_gb": 200, "shuffle_ratio": 0.5, "spill_gb": 0},
    "log_parse": {"data_gb": 2000, "shuffle_ratio": 1.0, "spill_gb": 600},
}

picks = {}
for name, workload in workloads.items():
    picks[name] = pick_node_family(workload)
    print(f"{name:<14} -> {picks[name]}")

assert picks["nightly_etl"].startswith("memory-optimized")
assert picks["feature_math"].startswith("compute-optimized")
assert picks["log_parse"].startswith("storage-optimized")


### Try it 2. Compute DBU cost and compare job with all-purpose

**Task.** A job uses 8 workers plus a driver, at 4 DBUs per node-hour, and runs for 45 minutes. Compute the DBUs and the cost at an illustrative job rate of 0.40 per DBU and an all-purpose rate of 1.00 per DBU, then report the percentage saved by running on jobs compute.

**Hint.** `dbu_per_hour = (workers + 1) x DBUs per node-hour`, and `dbus = dbu_per_hour x runtime_hours`.

**Expected result.** 36 DBUs per hour, 27 DBUs for the run, 10.80 on jobs compute versus 27.00 on all-purpose, a saving of 60 percent.

In [ ]:
DBUS_PER_NODE_HOUR = 4.0
WORKERS = 8
RUNTIME_MINUTES = 45
JOB_PRICE_PER_DBU = 0.40          # illustrative example rates
ALL_PURPOSE_PRICE_PER_DBU = 1.00

nodes = WORKERS + 1
dbu_per_hour = nodes * DBUS_PER_NODE_HOUR
runtime_hours = RUNTIME_MINUTES / 60
dbus = dbu_per_hour * runtime_hours

job_cost = dbus * JOB_PRICE_PER_DBU
all_purpose_cost = dbus * ALL_PURPOSE_PRICE_PER_DBU
savings_pct = (all_purpose_cost - job_cost) / all_purpose_cost

print(f"{nodes} nodes x {DBUS_PER_NODE_HOUR} DBU/h = {dbu_per_hour:.0f} DBU/h")
print(f"{RUNTIME_MINUTES} min -> {dbus:.1f} DBUs")
print(f"jobs compute: {job_cost:.2f}")
print(f"all-purpose:  {all_purpose_cost:.2f}  (saving {savings_pct * 100:.0f}%)")

assert dbu_per_hour == 36.0
assert round(dbus, 2) == 27.0
assert round(job_cost, 2) == 10.80
assert round(all_purpose_cost, 2) == 27.00
assert round(savings_pct * 100, 1) == 60.0


### Try it 3. Decide partitioning versus liquid clustering

**Task.** Given a table size and the filter rate and cardinality of each column, return a layout plan. Partition only when the table is at least 100 GB and some column is filtered by at least 90 percent of queries with cardinality at most 10,000. Leave tables under 50 GB alone. Otherwise use liquid clustering on the two most-filtered columns.

**Hint.** Sort the columns by `filter_rate` to pick the clustering keys, and check the low-cardinality always-filtered condition first.

**Expected result.** The 900 GB events table partitions by `event_date`, the 900 GB clicks table clusters on `(order_ts, customer_id)`, and the 5 GB table gets no layout change.

In [ ]:
def choose_layout(size_gb, columns):
    low_cardinality_always = [
        name for name, stats in columns.items()
        if stats["filter_rate"] >= 0.9 and stats["cardinality"] <= 10_000
    ]
    if size_gb >= 100 and low_cardinality_always:
        return "partition by " + low_cardinality_always[0]
    if size_gb < 50:
        return "plain Delta, no layout change"
    top_two = sorted(columns, key=lambda name: columns[name]["filter_rate"], reverse=True)[:2]
    return "liquid clustering on (" + ", ".join(top_two) + ")"


events = {
    "event_date": {"cardinality": 730, "filter_rate": 1.0},
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.7},
    "country": {"cardinality": 40, "filter_rate": 0.1},
}
clicks = {
    "order_ts": {"cardinality": 8_000_000, "filter_rate": 0.9},
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.8},
    "country": {"cardinality": 40, "filter_rate": 0.05},
}
tiny = {
    "event_date": {"cardinality": 730, "filter_rate": 1.0},
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.7},
}

plan_events = choose_layout(900, events)
plan_clicks = choose_layout(900, clicks)
plan_tiny = choose_layout(5, tiny)

print("events (900 GB):", plan_events)
print("clicks (900 GB):", plan_clicks)
print("tiny (5 GB):    ", plan_tiny)

assert plan_events == "partition by event_date"
assert plan_clicks == "liquid clustering on (order_ts, customer_id)"
assert plan_tiny == "plain Delta, no layout change"


### Try it 4. Detect skew and size a salt

**Task.** Given the row counts of the partitions in a shuffle stage, decide whether the stage is skewed. Treat a partition as skewed when its row count is at least three times the median. Then work out how many salt buckets the hot key needs so each bucket is no larger than a target number of rows.

**Hint.** Use `statistics.median` for the threshold and `math.ceil(max_rows / target_rows)` for the bucket count, capped at 256.

**Expected result.** The skewed example has a median of 100.5 and is detected; the balanced example is not; the hot key needs 3 buckets at a target of 5,000 rows.

In [ ]:
import math
import statistics


def detect_skew(partition_rows, factor=3.0):
    median = statistics.median(partition_rows)
    return max(partition_rows) / median >= factor, median


def salt_buckets(max_rows, target_rows, cap=256):
    return max(1, min(cap, math.ceil(max_rows / target_rows)))


skewed = [100, 98, 105, 101, 99, 100, 12_000, 103]
balanced = [100, 98, 105, 101, 99, 100, 103, 104]

skewed_flag, skewed_median = detect_skew(skewed)
balanced_flag, _ = detect_skew(balanced)
buckets = salt_buckets(max(skewed), target_rows=5000)

print(f"skewed median = {skewed_median:.1f}, max/median = {max(skewed) / skewed_median:.1f}")
print("skewed detected:", skewed_flag)
print("balanced is skewed:", balanced_flag)
print("salt buckets for the hot key:", buckets)

assert skewed_median == 100.5
assert skewed_flag is True
assert balanced_flag is False
assert buckets == 3


### Try it 5. Choose a target file count from table size

**Task.** A table is 2 TB and the target file size is 256 MB. Compute how many files it should have. Then, for a table currently split into 200,000 files averaging 10 MB each, decide whether it needs compaction, treating an average below half the target size as too small.

**Hint.** One TB is 1024 x 1024 MB. Compact when the current average file size is below `target_mb / 2`.

**Expected result.** 8,192 files at 256 MB each, a current average of 10 MB, and compaction is needed.

In [ ]:
import math

TARGET_FILE_MB = 256
TABLE_TB = 2.0
MB_PER_TB = 1024 * 1024

target_files = math.ceil(TABLE_TB * MB_PER_TB / TARGET_FILE_MB)

current_files = 200_000
current_avg_mb = (current_files * 10) / current_files
needs_compaction = current_avg_mb < TARGET_FILE_MB / 2

print(f"2 TB at {TARGET_FILE_MB} MB per file -> {target_files} files")
print(f"current average file size = {current_avg_mb:.0f} MB")
print("needs compaction:", needs_compaction)

assert target_files == 8192
assert current_avg_mb == 10.0
assert needs_compaction is True


### Try it 6. Compare a Photon run with a non-Photon run

**Task.** A job takes 60 minutes on a normal cluster at 32 DBUs per hour. With Photon it takes 24 minutes but consumes 30 percent more DBUs per hour. Compute both costs at 0.15 per DBU, the speedup, and whether Photon is cheaper per run.

**Hint.** Cost is `dbu_per_hour x minutes / 60 x price`. Photon can cost less per job even though it costs more per hour.

**Expected result.** 4.80 without Photon, 2.50 with Photon, a 2.50x speedup, and Photon is cheaper for this job.

In [ ]:
NON_PHOTON_DBU_PER_HOUR = 32.0
PHOTON_DBU_PER_HOUR = 41.6        # 30 percent more DBUs per hour
NON_PHOTON_MINUTES = 60
PHOTON_MINUTES = 24
PRICE_PER_DBU = 0.15


def job_cost(dbu_per_hour, minutes, price_per_dbu):
    return dbu_per_hour * (minutes / 60) * price_per_dbu


before = job_cost(NON_PHOTON_DBU_PER_HOUR, NON_PHOTON_MINUTES, PRICE_PER_DBU)
after = job_cost(PHOTON_DBU_PER_HOUR, PHOTON_MINUTES, PRICE_PER_DBU)
speedup = NON_PHOTON_MINUTES / PHOTON_MINUTES

print(f"without Photon: {before:.2f} for {NON_PHOTON_MINUTES} min")
print(f"with Photon:    {after:.2f} for {PHOTON_MINUTES} min")
print(f"speedup = {speedup:.2f}x, Photon is cheaper per run = {after < before}")

assert round(before, 2) == 4.80
assert round(after, 2) == 2.50
assert speedup == 2.5
assert after < before


### Try it 7. Estimate an autoscaling range from concurrency

**Task.** A stage has 2,000 shuffle tasks at 2 minutes each and must finish in 20 minutes. With 8 cores per worker, compute the concurrent slots needed, the steady worker count, and an autoscaling range: minimum at 20 percent of steady (floor of 2) and maximum at 20 percent headroom over steady.

**Hint.** `slots = ceil(total_task_minutes / target_minutes)`, then `workers = ceil(slots / cores_per_worker)`.

**Expected result.** 4,000 task-minutes, 200 slots, 25 steady workers, and a range of 5 to 30 workers.

In [ ]:
import math

TASKS = 2000
MINUTES_PER_TASK = 2.0
TARGET_STAGE_MINUTES = 20
CORES_PER_WORKER = 8
HEADROOM = 1.2
MIN_FLOOR = 2

total_task_minutes = TASKS * MINUTES_PER_TASK
concurrent_slots = math.ceil(total_task_minutes / TARGET_STAGE_MINUTES)
steady_workers = math.ceil(concurrent_slots / CORES_PER_WORKER)
max_workers = math.ceil(steady_workers * HEADROOM)
min_workers = max(MIN_FLOOR, math.ceil(steady_workers * 0.2))

print(f"total task minutes = {total_task_minutes:.0f}")
print(f"concurrent slots needed = {concurrent_slots}")
print(f"steady workers = {steady_workers}")
print(f"autoscaling range = {min_workers} to {max_workers}")

assert concurrent_slots == 200
assert steady_workers == 25
assert min_workers == 5
assert max_workers == 30


## Real world use cases for Databricks

Read each case as an answer template rather than a story. The shape is always the same: what arrives, which layer handles it, which Databricks feature does the hard part, and why the alternative would be worse. When an interviewer asks "tell me about a lakehouse you built", the useful content is the medallion boundary, the ingestion mode, the governance decision, and the one number that proved it worked.

### 1. Retail lakehouse: raw events to executive dashboards

```text
S3 raw JSON -> Auto Loader (bronze) -> dedupe/type (silver) -> fct/dim (gold) -> SQL warehouse dashboards
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
@dlt.table(name="gold_daily_revenue")
def gold_daily_revenue():
    return (
        dlt.read("silver_orders")
        .groupBy("order_date", "country")
        .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))
    )

Why Databricks: one pipeline feeds BI and data science, Delta gives ACID merges and time travel, and Unity Catalog keeps PII governed with masks and row filters.

### 2. Streaming CDC from operational databases

Debezium/Kafka lands change events; a streaming job merges them into Delta silver tables for near-real-time reporting.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "cdc.public.orders")
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"))
    .select("o.*")
    .writeStream
    .option("checkpointLocation", "/Volumes/acme/checkpoints/cdc_orders")
    .trigger(processingTime="1 minute")
    .foreachBatch(merge_into_silver)
    .start()
)

Why Databricks: structured streaming plus Delta MERGE gives exactly-once ingestion with upserts and deletes in the same platform as batch.

### 3. Migrating an on-prem Hive warehouse

Terabytes of Hive tables move to ADLS/S3, are upgraded to Delta, and Oozie jobs become Databricks Workflows.

Why Databricks: the Spark migration path is direct, Unity Catalog replaces Hive metastore governance, and liquid clustering plus Photon usually beats the old cluster.

### 4. Machine-learning platform with one feature definition

Feature tables in Delta are built nightly, registered in Unity Catalog, and consumed by MLflow-tracked training jobs; serving reads the same features from an online store.

Why Databricks: one governed copy of data, lineage from table to model, and MLflow integrated with the same catalog.

### 5. Cost-optimization program for an existing workspace

The platform team moves all scheduled work to job clusters with spot fallback, enables Photon and liquid clustering on hot tables, rightsizes SQL warehouses, and reviews `system.billing.usage` monthly by cost-center tag.

Why Databricks: cost is controllable when workload isolation, cluster policy, and table layout are treated as engineering, not afterthoughts.

If you can explain the compute plane and billing split, choose a table layout from the query pattern, read `system.billing.usage` and `system.query.history` to prove a claim, and design a task DAG that is idempotent and repairable, you can answer most Databricks interviews. The exercises above are the arithmetic behind those answers; the sections before them are the mechanics. Rehearse the mechanism first, then the failure mode, then the measurement that would confirm it.